# Gold Layer — Business Metrics & KPI Aggregations

**Project:** Bank Customer Churn Analytics (Medallion Architecture)  
**Notebook:** `03_gold_kpi_tables`  
**Author:** Harshkumar Patel  
**Last Updated:** July 3, 2026

---

## Overview

This notebook implements the **Gold layer** of the Medallion Architecture.

The Silver Star Schema is transformed into business-ready KPI tables using Spark SQL aggregations. These tables are optimized for reporting and serve as the primary data source for the Power BI dashboard.

---

## Responsibilities

- Load the Silver Star Schema
- Create a unified analytical view
- Generate business KPI tables
- Store aggregated results as Delta tables
- Validate the output datasets

> **Gold Principle:** Aggregate and model data for business consumption. All outputs are optimized for reporting and decision-making.

---

## Source

| Property | Value |
|----------|-------|
| Database | `churn_silver` |
| Source | Star Schema |
| Fact Tables | 1 |
| Dimension Tables | 3 |
| Records | 80,000 |

---

## Target Tables

| Table | Purpose |
|--------|---------|
| `gold_churn_summary` | Executive KPIs |
| `gold_churn_by_segment` | Customer segmentation metrics |
| `gold_churn_by_reason` | Inferred churn drivers |
| `gold_churn_risk_analysis` | Behavioral risk validation |
| `gold_churn_by_occupation` | Occupation-based churn analysis |
| `gold_churn_active_vs_inactive` | Membership activity analysis |
| `gold_churn_by_tenure` | Customer tenure analysis |
| `gold_churn_top_provinces` | Geographic churn performance |

---

## Pipeline Flow

```
Silver Star Schema
        │
        ▼
Create Master View
        │
        ▼
Business Aggregations
        │
        ▼
Gold Delta Tables
        │
        ▼
Power BI Dashboard
```

## Step 1 — Load Silver Tables

Load the Silver fact and dimension tables into Spark DataFrames for Gold layer aggregation.

In [0]:
# loading data from the silver layer star schema tables into gold for aggregation
df_fact = spark.table("churn_silver.fact_churn_events")
df_customer = spark.table("churn_silver.dim_customer")
df_scores = spark.table("churn_silver.dim_scores")
df_features = spark.table("churn_silver.dim_customer_features")

print(f"fact rows: {df_fact.count():,}")

## Step 2 — Create Analytical Master View

Create a reusable SQL view by joining the Silver fact and dimension tables. This consolidated view serves as the foundation for all Gold layer KPI calculations and eliminates repetitive join logic.

In [0]:
%sql
CREATE OR REPLACE VIEW churn_silver.vw_churn_master AS
SELECT
    f.*,
    c.credit_score,
    c.gender,
    c.age,
    c.occupation,
    c.origin_province,
    c.customer_segment,
    s.engagement_score,
    s.risk_score,
    ft.age_group,
    ft.balance_tier,
    ft.tenure_group,
    ft.engagement_tier,
    ft.churn_risk_label,
    ft.churn_reason_inferred
FROM churn_silver.fact_churn_events f
JOIN churn_silver.dim_customer c ON f.id = c.id
JOIN churn_silver.dim_scores s ON f.id = s.id
JOIN churn_silver.dim_customer_features ft ON f.id = ft.id

In [0]:
%sql
SELECT * FROM churn_silver.vw_churn_master
LIMIT 5


## Step 3 — Generate Executive KPI Tables

Create aggregated Delta tables using Spark SQL to support executive reporting and interactive Power BI dashboards.

### gold_churn_summary

Generate overall business KPIs for executive dashboard cards.

In [0]:
%sql
CREATE OR REPLACE TABLE churn_gold.gold_churn_summary
USING DELTA
AS
SELECT
    COUNT(*)                                                         AS total_customers,
    SUM(CASE WHEN exit = true THEN 1 ELSE 0 END)                    AS total_churned,
    ROUND(SUM(CASE WHEN exit = true THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS churn_rate_pct,
    ROUND(AVG(balance), 2)                                           AS avg_balance,
    ROUND(AVG(engagement_score), 2)                                  AS avg_engagement_score,
    ROUND(AVG(risk_score), 2)                                        AS avg_risk_score
FROM churn_silver.vw_churn_master

In [0]:
%sql
SELECT * FROM churn_gold.gold_churn_summary

In [0]:
%sql
SELECT churn_reason_inferred, COUNT(*) 
FROM churn_silver.vw_churn_master 
GROUP BY churn_reason_inferred

### gold_churn_by_segment

Aggregate churn performance across key customer segments including age, balance, and engagement.

In [0]:
%sql
CREATE OR REPLACE TABLE churn_gold.gold_churn_by_segment
USING DELTA
AS

WITH age_segment AS (
    SELECT
        'age_group'                AS segment_type,
        age_group                  AS segment_value,
        COUNT(*)                   AS total_customers,
        SUM(CASE WHEN exit = true THEN 1 ELSE 0 END) AS churned,
        ROUND(SUM(CASE WHEN exit = true THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS churn_rate_pct
    FROM churn_silver.vw_churn_master
    GROUP BY age_group
),
balance_segment AS (
    SELECT
        'balance_tier',
        balance_tier,
        COUNT(*),
        SUM(CASE WHEN exit = true THEN 1 ELSE 0 END),
        ROUND(SUM(CASE WHEN exit = true THEN 1 ELSE 0 END) / COUNT(*) * 100, 2)
    FROM churn_silver.vw_churn_master
    GROUP BY balance_tier
),
engagement_segment AS (
    SELECT
        'engagement_tier', engagement_tier,
        COUNT(*),
        SUM(CASE WHEN exit = true THEN 1 ELSE 0 END),
        ROUND(SUM(CASE WHEN exit = true THEN 1 ELSE 0 END) / COUNT(*) * 100, 2)
    FROM churn_silver.vw_churn_master
    GROUP BY engagement_tier
)

SELECT * FROM age_segment
UNION ALL
SELECT * FROM balance_segment
UNION ALL
SELECT * FROM engagement_segment

In [0]:
%sql
SELECT * FROM churn_gold.gold_churn_by_segment
ORDER BY churn_rate_pct DESC

### gold_churn_by_reason

Summarize inferred customer churn drivers derived during the Silver transformation process.

In [0]:
%sql
CREATE OR REPLACE TABLE churn_gold.gold_churn_by_reason
USING DELTA
AS
SELECT
    churn_reason_inferred,
    COUNT(*)        AS total_customers,
    SUM(CASE WHEN exit = true THEN 1 ELSE 0 END) AS churned,
    ROUND(SUM(CASE WHEN exit = true THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS churn_rate_pct
FROM churn_silver.vw_churn_master
GROUP BY churn_reason_inferred
ORDER BY churned DESC

In [0]:
%sql
SELECT * FROM churn_gold.gold_churn_by_reason
ORDER BY churned DESC

### gold_churn_risk_analysis

Evaluate the relationship between the engineered behavioral risk labels and observed customer churn.

In [0]:
%sql
CREATE OR REPLACE TABLE churn_gold.gold_churn_risk_analysis
USING DELTA
AS
SELECT
    churn_risk_label,
    COUNT(*)                                                           AS total_customers,
    SUM(CASE WHEN exit = true THEN 1 ELSE 0 END)                     AS churned,
    ROUND(SUM(CASE WHEN exit = true THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS churn_rate_pct,
    ROUND(AVG(engagement_score), 2)                                   AS avg_engagement_score,
    ROUND(AVG(risk_score), 2)                                         AS avg_risk_score
FROM churn_silver.vw_churn_master
GROUP BY churn_risk_label
ORDER BY churn_rate_pct DESC

In [0]:
%sql
SELECT * FROM churn_gold.gold_churn_risk_analysis
ORDER BY churned DESC

### gold_churn_by_occupation

Analyze churn behavior across customer occupations.

In [0]:
%sql
CREATE OR REPLACE TABLE churn_gold.gold_churn_by_occupation
USING DELTA
AS
SELECT
    occupation,
    COUNT(*)        AS total_customers,
    SUM(CASE WHEN exit = true THEN 1 ELSE 0 END) AS churned,
    ROUND(SUM(CASE WHEN exit = true THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS churn_rate_pct,
    ROUND(AVG(engagement_score), 2) AS avg_engagement_score
FROM churn_silver.vw_churn_master
GROUP BY occupation
ORDER BY churn_rate_pct DESC

In [0]:
%sql
SELECT * FROM churn_gold.gold_churn_by_occupation
ORDER BY churned DESC

### gold_churn_active_vs_inactive

Compare churn metrics between active and inactive customers.

In [0]:
%sql
CREATE OR REPLACE TABLE churn_gold.gold_churn_active_vs_inactive
USING DELTA
AS
SELECT
    CASE WHEN active_member = true THEN 'Active' ELSE 'Inactive' END AS member_status,
    COUNT(*)        AS total_customers,
    SUM(CASE WHEN exit = true THEN 1 ELSE 0 END) AS churned,
    ROUND(SUM(CASE WHEN exit = true THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS churn_rate_pct,
    ROUND(AVG(engagement_score), 2) AS avg_engagement_score
FROM churn_silver.vw_churn_master
GROUP BY active_member
ORDER BY churn_rate_pct DESC

In [0]:
%sql
SELECT * FROM churn_gold.gold_churn_active_vs_inactive
ORDER BY churned DESC

### gold_churn_by_tenure

Measure customer churn across tenure segments.

In [0]:
%sql
CREATE OR REPLACE TABLE churn_gold.gold_churn_by_tenure
USING DELTA
AS
SELECT
    tenure_group,
    COUNT(*)        AS total_customers,
    SUM(CASE WHEN exit = true THEN 1 ELSE 0 END) AS churned,
    ROUND(SUM(CASE WHEN exit = true THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS churn_rate_pct,
    ROUND(AVG(engagement_score), 2) AS avg_engagement_score
FROM churn_silver.vw_churn_master
GROUP BY tenure_group
ORDER BY churn_rate_pct DESC

In [0]:
%sql
SELECT * FROM churn_gold.gold_churn_by_tenure
ORDER BY churned DESC

In [0]:
### gold_churn_top_provinces

Rank provinces by customer churn rate to identify high-risk geographic regions.

In [0]:
%sql
CREATE OR REPLACE TABLE churn_gold.gold_churn_top_provinces
USING DELTA
AS
WITH province_churn AS (
    SELECT
        origin_province,
        COUNT(*)        AS total_customers,
        SUM(CASE WHEN exit = true THEN 1 ELSE 0 END) AS churned,
        ROUND(SUM(CASE WHEN exit = true THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS churn_rate_pct
    FROM churn_silver.vw_churn_master
    GROUP BY origin_province
)
SELECT
    origin_province,
    total_customers,
    churned,
    churn_rate_pct,
    RANK() OVER (ORDER BY churn_rate_pct DESC) AS churn_rank
FROM province_churn
ORDER BY churn_rank

In [0]:
%sql
SELECT * FROM churn_gold.gold_churn_top_provinces
ORDER BY churned DESC

## Gold Layer Summary

| Output | Result |
|---------|--------|
| Source Tables | 4 |
| Master View | 1 |
| Gold Tables Created | 8 |
| Storage Format | Delta Lake |
| Query Engine | Spark SQL |

### Deliverables

- Executive KPI summary
- Customer segmentation metrics
- Churn reason analysis
- Behavioral risk validation
- Occupation analysis
- Membership activity analysis
- Tenure analysis
- Geographic churn rankings

### Status

Gold layer completed successfully.

The resulting Delta tables are optimized for consumption by the Power BI churn analytics dashboard.